# 04 · Agent Tools: MCP and A2A

> **The story:** The Rules agent knows the law, but Sara's work happens in Outlook. Two more agents read her mail and manage her calendar, as her, and ask before doing anything.

**You will learn**
- MCP tools from the Foundry catalog: Work IQ Mail and Work IQ Calendar
- Delegated access: these agents act **as you**, with your Microsoft 365 permissions
- Least privilege: each agent only gets the tools it needs (the Mail agent can't forward or delete)
- Approval before every write: Foundry pauses the tool call until a person decides
- A2A: any A2A-compatible agent can hand a question to the Rules agent

**Time:** about 20 minutes · **Before you start:**
- Notebooks 00 to 02 finished
- A Microsoft 365 Copilot licence (otherwise watch the facilitator)
- The portal step below

<sub>Workshop material: the rental documents summarise Dubai rules for training only. Not legal advice.</sub>

> ### 👉 Follow along in the portal
>
> **One-time step: connect the Work IQ tools (consent needs a person, so it's portal-only).**
> 1. ai.azure.com → your project → **Build → Tools → Connect a tool → Catalog**.
> 2. Search **Work IQ Mail** → **Create** → connection name **`WorkIQMail`** (exactly this) → sign in and accept.
> 3. Repeat for **Work IQ Calendar** → connection name **`WorkIQCalendar`**.

In [ ]:
# @title Setup: run this first (about 1 minute, then sign in)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()

settings = WorkshopEnvironment.discover(credential)
print(f"✅ Signed in. Your environment: {settings.resource_group}")

## Step 1 · Check the connections

In [ ]:
from azure.ai.projects import AIProjectClient

connections = {c.name: c for c in AIProjectClient(settings.project_endpoint, credential).connections.list()}
missing = [n for n in (settings.mail_connection_name, settings.calendar_connection_name) if n not in connections]
if missing:
    raise RuntimeError(f"Do the portal step above first. Missing connection(s): {missing}")
table([{"connection": n, "target": connections[n].target} for n in (settings.mail_connection_name, settings.calendar_connection_name)])

## Step 2 · Create the Mail and Calendar agents

Each agent gets **only** the tools in this table. Reads run freely; every write pauses for approval.

In [ ]:
from dld_workshop.agents import (create_mail_and_calendar_agents, MAIL_AGENT, CALENDAR_AGENT,
                                 MAIL_READ, MAIL_WRITE, CALENDAR_READ, CALENDAR_WRITE)

versions = create_mail_and_calendar_agents(settings, credential)
table([
    {"agent": MAIL_AGENT, "version": versions[MAIL_AGENT], "reads (no approval)": ", ".join(MAIL_READ),
     "writes (approval required)": ", ".join(MAIL_WRITE), "not available at all": "forward, delete, send new mail"},
    {"agent": CALENDAR_AGENT, "version": versions[CALENDAR_AGENT], "reads (no approval)": ", ".join(CALENDAR_READ),
     "writes (approval required)": ", ".join(CALENDAR_WRITE), "not available at all": "forward, accept/decline, rooms"},
])

## Step 3 · Play the tenant

Send yourself this email from Outlook, then run the next cell.

In [ ]:
import json
EMAILS = json.loads((ROOT / "workshop/data/tenant_emails.json").read_text())
email = EMAILS["genuine"]
print("Send this email TO YOURSELF from Outlook (copy and paste):\n")
print("Subject:", email["subject"])
print("-" * 60)
print(email["body"])

## Step 4 · The Mail agent reads it (as you)

Reading needs no approval.

In [ ]:
from dld_workshop.agents import run_agent

read = run_agent(settings, credential, MAIL_AGENT, "Find the newest rent email and give me the sender, received time, subject and full text.")
show("Mail agent", read["text"])

## Step 5 · The Calendar agent proposes an appointment

It checks your calendar for a free slot, then tries to create the event. Foundry **pauses** the write and returns an approval request instead.

In [ ]:
booking = run_agent(settings, credential, CALENDAR_AGENT,
                    "Book a mediation information appointment for a rent increase enquiry in the next free slot.")
table([{"agent": p["agent"], "wants to": p["tool"], "details": str(p["arguments"])[:300]} for p in booking["pending"]])
show("Calendar agent", booking["text"] or "(waiting for your decision)")

### Your decision
Set `APPROVE_BOOKING = True` to let the event be created in your calendar, or leave it `False` to reject it. Either way the agent is told your decision.

In [ ]:
from dld_workshop.agents import decide

APPROVE_BOOKING = False   # ← change to True to create the event
if booking["pending"]:
    booking = decide(settings, credential, booking["pending"], approve=APPROVE_BOOKING)
show("After your decision", booking["text"])

Clean up: ask the Calendar agent to delete the test event. Deleting is a write too, so it asks again.

In [ ]:
cleanup = run_agent(settings, credential, CALENDAR_AGENT,
                    "Delete the 'DLD Rental Mediation Information - Rent increase enquiry' event you booked in the next 7 days, if there is one.")
if cleanup["pending"]:
    cleanup = decide(settings, credential, cleanup["pending"], approve=APPROVE_BOOKING)
show("Calendar agent", cleanup["text"])

> ### 👉 Follow along in the portal
>
> **Agents → `DLD-Mail`** → **playground**: ask "find the newest rent email". Then ask it to reply: the playground shows the same **approval** prompt with Approve / Deny buttons.
> **Build → Tools**: your two Work IQ connections. Each one acts as the person who signed in.

## Step 6 · Agent-to-agent (A2A)

The Rules agent gets a public "business card" that any A2A-compatible agent can read; then we send it a question using only the open protocol, not the Foundry SDK.

In [ ]:
import requests
from dld_workshop.agents import enable_rules_a2a, send_a2a_message, RULES_AGENT

enable_rules_a2a(settings, credential)
token = credential.get_token("https://ai.azure.com/.default").token
card = requests.get(f"{settings.project_endpoint}/agents/{RULES_AGENT}/endpoint/protocols/a2a/agentCard/v1.0",
                    headers={"Authorization": f"Bearer {token}"}, timeout=60)
card.raise_for_status()
table([{"field": k, "value": str(card.json().get(k))[:120]} for k in ["name", "description", "version", "skills"]])

task = send_a2a_message(settings, credential, "How much notice must a Dubai landlord give before changing the rent? One sentence.")["result"]["task"]
show(f"A2A reply (task state: {task['status']['state']})", task["artifacts"][0]["parts"][0]["text"])

---
## What to notice
- The Mail and Calendar agents act as you: they see only your mailbox and calendar.
- Writes never happen silently: Foundry pauses the exact tool call until you decide.
- **MCP** plugs an agent into a tool; **A2A** hands a task to another agent; **Foundry IQ** is the knowledge layer.

**Try it yourself:** in the `DLD-Mail` playground, ask it to forward the email to someone. It can't: the tool isn't there.